#### Porcentaje de veces que se escogió cada clave 

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
def plot_key_proportions_with_pruning(csv_path, dataset_name):
    # leaf,kernel,mode,radius,count,total,key
    df = pd.read_csv(csv_path)
    
    # 1. Calculamos el porcentaje de puntos PODADOS
    # (Total puntos en la hoja - Puntos dentro del rango) / Total puntos en la hoja
    # Multiplicamos por 100 para tener el porcentaje
    df['pruning_pct'] = (df['total'] - df['count']) / df['total'] * 100
    
    # Mapeo de claves
    key_map = {0: 'K0 (Azimuthal)', 1: 'K1 (Rxy/Theta)', 2: 'K2 (Z/Rxyz)'}
    df['key_name'] = df['key'].map(key_map)

    # 2. Creamos la gráfica
    plt.figure(figsize=(14, 8))
    ax = sns.countplot(
        data=df, 
        x='radius', 
        hue='key_name', 
        palette='viridis'
    )

    # 3. Calculamos la media de poda para cada barra y la añadimos como etiqueta
    # Agrupamos por los mismos criterios que la gráfica (radius y key_name)
    means = df.groupby(['radius', 'key_name'])['pruning_pct'].mean().reset_index()

    # Iteramos sobre las barras generadas por Seaborn
    # Importante: Seaborn dibuja las barras en el mismo orden que el hue
    for i, container in enumerate(ax.containers):
        # Obtenemos la clave (K0, K1 o K2) que corresponde a este grupo de barras (hue)
        current_key = key_map[i]
        
        # Filtramos las medias para este grupo
        group_means = means[means['key_name'] == current_key]['pruning_pct'].values
        
        # Añadimos el texto sobre cada barra
        # labels = [f'{val:.1f}%' for val in group_means]
        # Si algunas barras no existen para ciertos radios, controlamos el índice
        for j, bar in enumerate(container):
            if j < len(group_means):
                height = bar.get_height()
                if height > 0: # Solo poner etiqueta si la barra existe
                    ax.text(
                        bar.get_x() + bar.get_width() / 2, 
                        height + (df.shape[0]*0.01), # Un poco por encima de la barra
                        f'{group_means[j]:.1f}%', 
                        ha='center', va='bottom', 
                        fontsize=9, fontweight='bold', rotation=0
                    )

    plt.title(f'Distribución de Claves y % Medio de Poda - Dataset: {dataset_name}', fontsize=14)
    plt.xlabel('Radio de Búsqueda')
    plt.ylabel('Cantidad de Hojas')
    plt.legend(title='Clave y Poda Media')
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()

OSError: [WinError 1] Función incorrecta: '\\\\wsl.localhost\\Ubuntu\\home\\manu\\proyecto_tfg\\octrees-coordenadas-polares-v2'

In [ ]:
datasets = [
    "Lille_0.log",
    "Paris_Luxembourg_6.log"
]
for dataset in datasets:
    plot_key_proportions_with_pruning(f'logs/v1.1/ranges/{dataset}', dataset.split('.')[0])

OSError: [Errno 22] Invalid argument

### Tiempo invertido en calcular rangos seleccionados vs loop de comprobación de vecindades

In [ ]:
def plot_detailed_time_breakdown(csv_path, filename):
    # Supongamos que ahora el CSV tiene: dataset,kernel,radius,mode,get_range_time,loop_time
    df = pd.read_csv(csv_path)
    
    # 1. Agrupamos por las variables que definen la "dificultad" de la búsqueda
    # Calculamos la media para cada combinación
    analysis = df.groupby(['kernel','radius', 'get_range'])[['get_range_time', 'loop_time']].mean().reset_index()
    
    # 2. Creamos una visualización comparativa por Radio
    # Esto te permitirá ver en qué radio empieza a ser rentable la optimización
    plt.figure(figsize=(12, 7))
    sns.barplot(data=analysis, x='radius', y='loop_time', hue='mode')
    plt.title('Tiempo de Bucle (Filtrado) según Radio y Modo')
    plt.ylabel('Nanosegundos')
    plt.show()

    # 3. Visualización del Overhead (Tiempo de decisión)
    plt.figure(figsize=(12, 7))
    sns.barplot(data=analysis[analysis['mode'] != 'none'], x='radius', y='get_range_time', hue='mode')
    plt.title('Overhead de Selección de Rango según Radio - Dataset: ' + filename)
    plt.ylabel('Nanosegundos')
    plt.show()

#### Reordenamiento esférico

In [ ]:
datasets = [
    "Lille_0_spherical.csv",
    "Paris_Luxembourg_6_spherical.csv",
    "bildstein_station1_xyz_intensity_rgb_spherical.csv",
    "sg27_station8_intensity_rgb_spherical.csv"
]
for dataset in datasets:
    plot_detailed_time_breakdown(f'logs/v1.1/{dataset}', dataset.split('_spherical')[0])

#### Reordenamiento cilíndrico

In [ ]:
datasets = [
    "Lille_0_cylindrical.csv",
    "Paris_Luxembourg_6_cylindrical.csv",
    "bildstein_station1_xyz_intensity_rgb_cylindrical.csv",
    "sg27_station8_intensity_rgb_cylindrical.csv"
]
for dataset in datasets:
    plot_detailed_time_breakdown(f'logs/v1.1/{dataset}', dataset.split('_cylindrical')[0])